# OSRS Grand Exchange Price Predictor & Flipping Screener

This turnkey Google Colab notebook demonstrates the complete end-to-end pipeline:
1. Setup & Package Installation
2. Fetch real-time and historical data from the official OSRS Wiki API
3. Compute stationary features & calendar seasonality
4. Train a LightGBM time-series forecaster with walk-forward CV
5. Interactive Plotly visualizations
6. Live 2% tax-adjusted flipping screener

## Cell 1: Setup & Environment Installation

In [ ]:
# Install ge_predictor in editable/local mode or directly from repository
!pip install -q .

## Cell 2: Fetch Historical Data
Fetch 1-hour historical timeseries data for high-volume benchmark items: Shark (385), Prayer Potion (4) (139), and Cannonball (2).

In [ ]:
import pandas as pd
from ge_predictor.client import OSRSClient

client = OSRSClient(user_agent="GEPredictor-Colab - Contact: colab-user@example.com")

benchmark_items = {
    "Shark": 385,
    "Prayer Potion (4)": 139,
    "Cannonball": 2
}

data_dict = {}
for name, item_id in benchmark_items.items():
    ts = client.get_timeseries(item_id=item_id, timestep="1h")
    df = pd.DataFrame(ts)
    data_dict[name] = df
    print(f"Fetched {len(df)} 1-hour records for {name} (ID: {item_id})")

## Cell 3: Feature Engineering
Compute stationary features (log returns, EMA ratios, rolling volatility, spread, volume momentum) and inspect distributions.

In [ ]:
from ge_predictor.features import build_features, FEATURE_COLUMNS

# Process Shark features
shark_df = data_dict["Shark"]
shark_feat = build_features(shark_df, horizons=[1, 3])

print("Feature Matrix Summary for Shark:")
display(shark_feat[FEATURE_COLUMNS + ["target_forward_return_1"]].describe())

## Cell 4: Model Training
Train LightGBM using walk-forward TimeSeriesSplit and evaluate RMSE & Directional Accuracy.

In [ ]:
from ge_predictor.models import PricePredictor

predictor = PricePredictor(target_horizon=1)
metrics = predictor.fit_with_cv(shark_df, n_splits=5)

print("LightGBM Walk-Forward Evaluation Metrics:")
print(f"  Mean CV RMSE: {metrics['mean_cv_rmse']:.6f}")
print(f"  Mean CV Directional Accuracy: {metrics['mean_cv_directional_accuracy'] * 100:.2f}%")
print(f"  Total Training Samples: {metrics['n_samples']}")

## Cell 5: Interactive Visualization
Plot actual vs. predicted price movements along with net tax-adjusted margins using Plotly.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from ge_predictor.market_rules import calculate_ge_tax

# Clean features dataset for plotting
clean_df = predictor.prepare_data(shark_df)
X = clean_df[predictor.feature_cols]
preds = predictor.model.predict(X)
clean_df["pred_return"] = preds
clean_df["pred_mid_price"] = clean_df["mid_price"] * np.exp(preds)
clean_df["tax"] = clean_df["avgHighPrice"].apply(calculate_ge_tax)
clean_df["net_margin"] = (clean_df["avgHighPrice"] - clean_df["tax"]) - clean_df["avgLowPrice"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=("Price & Prediction", "Tax-Adjusted Net Margin (GP)"))

fig.add_trace(go.Scatter(x=clean_df["datetime"], y=clean_df["mid_price"], mode="lines", name="Actual Mid Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=clean_df["datetime"], y=clean_df["pred_mid_price"], mode="lines", name="Predicted Mid Price (1h Forward)", line=dict(dash="dot")), row=1, col=1)

fig.add_trace(go.Bar(x=clean_df["datetime"], y=clean_df["net_margin"], name="Net Margin"), row=2, col=1)

fig.update_layout(title="OSRS Shark Market Dynamics & Model Predictions", height=600)
fig.show()

## Cell 6: Live Tax-Adjusted Flipping Screener
Run the active flip screener for top liquidity items given a budget.

In [ ]:
budget_gp = 20_000_000  # @param {type:"integer"}
min_roi_pct = 1.0       # @param {type:"number"}

mapping = client.get_mapping()
latest = client.get_latest()

items_data = []
for m in mapping:
    item_id = m["id"]
    prices = latest.get(str(item_id)) or latest.get(item_id)
    if not prices:
        continue
    high = prices.get("high")
    low = prices.get("low")
    if not high or not low or low >= high:
        continue
    items_data.append({
        "id": item_id,
        "name": m.get("name"),
        "buy_price": low,
        "sell_price": high,
        "limit": m.get("limit") or 10000,
        "hourly_volume": 100.0
    })

from ge_predictor.scoring import find_top_flips
top_flips = find_top_flips(items_data, budget_gp=budget_gp, min_roi=min_roi_pct, top_n=10)

screener_df = pd.DataFrame(top_flips)
display(screener_df[["item_name", "buy_price", "sell_price", "tax_per_item", "net_margin", "roi_pct", "trade_quantity", "potential_4h_profit"]])